# Telefon değişince 16 bölgeli ve sade modellerin hatası
Aynı topraklar, aynı fotoğraflar ve aynı üç komşu kuralı ile iki yöntemi karşılaştırıyoruz. Ölçüm birimi toprak numunesi.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else next(p for p in [Path("work/soil-data"), Path("../work/soil-data")] if p.exists())
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)


def normalized_gray(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    return np.clip((gray - low) / max(high - low, 1.0), 0, 1)

def texture_at_six_scales(gray):
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        x = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        y = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((x + y) / 2)
    return values

cameras_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
feature_rows = []
for path in photos:
    phone, sample, _ = path.stem.rsplit("_", 2)
    patch = physical_patch(path, cameras_by_name[phone.lower()])
    gray = normalized_gray(patch)
    regions = [gray[y:y+100, x:x+100] for y in range(0, 400, 100) for x in range(0, 400, 100)]
    region_textures = np.array([texture_at_six_scales(region) for region in regions])
    spread = region_textures.std(axis=0)
    feature_rows.append([sample, *texture_at_six_scales(gray), *spread])
    if path.stem == "Motorola_Edge_H038_03":
        example = patch.copy()

scales = ["0.25", "0.5", "1", "2", "4", "8"]
columns = scales + ["sd_" + s for s in scales]
frame = pd.DataFrame(feature_rows, columns=["sample_id", *columns])
features = frame.groupby("sample_id").median().loc[labels.index]
assert features.shape == (24, 12) and np.isfinite(features).all().all()

## Bir telefonda öğrenip iki telefonda ölçme

Her telefon ve toprak için çekimlerin ortancasını alıyoruz. Her turda iki telefonda da bulunan 21 topraktan birini saklıyoruz. Ölçekleme ve üç komşu seçimi için kalan 20 toprağın yalnızca eğitim telefonuyla çekilmiş fotoğraflarını kullanıyoruz. Önce altı doku ölçüsünü, sonra bu altı ölçüyle birlikte 16 bölgedeki değişkenliği kullanıyoruz. Her model için iki tahminin laboratuvara olan hatalarını ve birbirlerine olan uzaklıklarını aynı EMD ölçeğinde hesaplıyoruz.

In [3]:
phones = ["Motorola_Edge", "Samsung_A52"]
frame["telefon"] = [p.stem.rsplit("_", 2)[0] for p in photos]
assert frame["sample_id"].tolist() == [p.stem.rsplit("_", 2)[1] for p in photos]
per_phone = frame.groupby(["telefon", "sample_id"])[columns].median()
motorola, samsung = [per_phone.loc[p].index for p in phones]
paired = labels.index[labels.index.isin(motorola) & labels.index.isin(samsung)]
assert len(paired) == 21 and paired.is_unique
widths = np.diff(np.log10(labels.columns.astype(float)))

def emd(a, b):
    return np.abs(a - b)[:-1] @ widths

records = []
for name, keys in [("Sade", scales), ("16 bölge", columns)]:
    for source, other in [phones, phones[::-1]]:
        for sample in paired:
            train = per_phone.loc[source, keys].loc[paired.drop(sample)]
            assert len(train) == 20 and sample not in train.index
            scaler = StandardScaler().fit(train)
            reference = scaler.transform(train)
            curves = []
            for viewed in [source, other]:
                query = per_phone.loc[viewed, keys].loc[[sample]]
                distance = np.linalg.norm(reference - scaler.transform(query), axis=1)
                nearest = train.index[np.argsort(distance, kind="stable")[:3]]
                curves.append(labels.loc[nearest].mean().to_numpy())
            truth = labels.loc[sample].to_numpy()
            same_error, cross_error = [emd(curve, truth) for curve in curves]
            records.append([name, source, sample, same_error, cross_error, cross_error - same_error, emd(*curves)])
results = pd.DataFrame(records, columns=["Model", "Eğitim telefonu", "Numune", "Aynı", "Diğer", "Hata farkı", "Tahminler arası EMD"])
assert len(results) == 84 and not results.duplicated(["Model", "Eğitim telefonu", "Numune"]).any()
metrics = ["Aynı", "Diğer", "Hata farkı", "Tahminler arası EMD"]
summary = results.groupby(["Model","Eğitim telefonu"])[metrics].mean()
assert np.allclose(summary.loc["Sade", ["Aynı", "Diğer"]], [[45.98,49.10],[45.01,45.26]], atol=0.005, rtol=0)
print(summary.round(3).to_string())
print("İki eğitim yönünün eşit ağırlıklı ortalaması")
print(results.groupby("Model")[metrics].mean().round(3).to_string())

                            Aynı   Diğer  Hata farkı  Tahminler arası EMD
Model    Eğitim telefonu                                                 
16 bölge Motorola_Edge    48.610  44.478      -4.132               21.955
         Samsung_A52      54.528  56.624       2.097               30.020
Sade     Motorola_Edge    45.985  49.095       3.111               19.912
         Samsung_A52      45.005  45.265       0.259               27.548
İki eğitim yönünün eşit ağırlıklı ortalaması
            Aynı   Diğer  Hata farkı  Tahminler arası EMD
Model                                                    
16 bölge  51.569  50.551      -1.018               25.987
Sade      45.495  47.180       1.685               23.730


In [4]:
print("Numune başına EMD")
print(results.pivot(index=["Numune", "Eğitim telefonu"], columns="Model", values=["Aynı", "Diğer"]).round(2).to_string())

Numune başına EMD
                           Aynı            Diğer        
Model                  16 bölge    Sade 16 bölge    Sade
Numune Eğitim telefonu                                  
F827   Motorola_Edge     127.07   50.31    90.73   62.26
       Samsung_A52       135.97   59.76    93.73   53.54
G190   Motorola_Edge      50.10   19.22    25.62   25.62
       Samsung_A52        23.34   27.19    74.97   23.61
H030   Motorola_Edge      21.38   30.27    30.27   30.27
       Samsung_A52        42.48   21.81    56.38   74.67
H037   Motorola_Edge      67.53   95.07   102.43   88.67
       Samsung_A52       127.70   49.55    97.39  108.26
H038   Motorola_Edge     128.23  168.25   100.74  129.21
       Samsung_A52       119.33  102.18   124.64  137.94
H126   Motorola_Edge      28.03   39.76    38.26   55.07
       Samsung_A52        28.03   78.11    26.00   26.00
H181   Motorola_Edge      19.27   19.27    17.02   19.06
       Samsung_A52        66.30   17.32    17.32   17.31
H183   Motoro